In [ ]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 24.1 MB/s eta 0:00:00


In [1]:
import os
import shutil
import fitz  # PyMuPDF
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path_to_root = "/content/drive/MyDrive/Finland"

In [4]:
folder_path = os.path.join(path_to_root,"data")

/content/drive/MyDrive/Finland/data


In [ ]:
def is_scanned(pdf_path):
    print(f"Checking file: {pdf_path}")
    try:
        doc = fitz.open(pdf_path)
        text_len = sum(len(p.get_text().strip()) for p in doc)
        image_count = sum(len(p.get_images()) for p in doc)
        result = text_len < 50 and image_count >= len(doc)
        print(f"Finished checking: {pdf_path} -> is_scanned={result}")
        return result
    except Exception as e:
        print(f"Error processing {pdf_path}: {e}")
        return None

def list_pdfs_recursive(folder):
    pdf_files = []
    for root, dirs, files in os.walk(folder):
        print(f"Entering folder: {root}")
        for file in files:
            if file.lower().endswith(".pdf"):
                full_path = os.path.join(root, file)
                print(f"Found PDF: {full_path}")
                pdf_files.append(full_path)
    print(f"Total PDFs found: {len(pdf_files)}")
    return pdf_files

print("Starting recursive PDF listing...")
pdf_paths = list_pdfs_recursive(folder_path)

print("\nStarting scan detection...\n")

data = []
for path in pdf_paths:
    scanned = is_scanned(path)
    data.append({
        "filename": path,
        "is_scanned": scanned
    })

df = pd.DataFrame(data, columns=["filename", "is_scanned"])

print("\nProcessing complete.")
print(df.head())

Starting recursive PDF listing...
Entering folder: /content/drive/MyDrive/research/Finland_data/data
Entering folder: /content/drive/MyDrive/research/Finland_data/data/Files_for_survey_487832
Found PDF: /content/drive/MyDrive/research/Finland_data/data/Files_for_survey_487832/00062_24-tilinpaatos_00-kkes-tilinpäätös-2023.pdf
Found PDF: /content/drive/MyDrive/research/Finland_data/data/Files_for_survey_487832/00019_03-tilinpaatos_00-stiftelsen-tre-smeder-bokslut-31.12.2023-underskrivet-(styrelsen).pdf
Found PDF: /content/drive/MyDrive/research/Finland_data/data/Files_for_survey_487832/00032_11-tilinpaatos_00-vuoden-2023-toimintakertomus-ja-konsernitilinpäätös-allekirjoitettu-hallitus.pdf
Found PDF: /content/drive/MyDrive/research/Finland_data/data/Files_for_survey_487832/00084_35-tilinpaatos_00-sakari-alhopuron-säätiö-tilinpäätös-liitetiedot-ja-tilintarkastuskertomus-2023.pdf
Found PDF: /content/drive/MyDrive/research/Finland_data/data/Files_for_survey_487832/00063_25-tilinp

In [ ]:
os.makedirs(os.path.join(path_to_root,"out"), exist_ok=True)
df.to_csv(os.path.join(os.path.join(path_to_root,"out"), "qc_scan_database.csv"), index=False)

In [ ]:
false_count = (df['is_scanned'] == False).sum()
print(f"Number of 'False' values in 'is_scanned' column: {false_count}")

Number of 'False' values in 'is_scanned' column: 204


In [ ]:
df.shape

(251, 2)

In [ ]:
# create folder if it doesn't exist
os.makedirs(os.path.join(path_to_root,"out/qc_scan_pass"), exist_ok=True)

for index, row in df.iterrows():
    filepath = row["filename"]
    scanned = row["is_scanned"]

    if scanned is False:
        try:
            filename = os.path.basename(filepath)
            destination = os.path.join(os.path.join(path_to_root,"out/qc_scan_pass"), filename)

            print(f"Copying: {filepath}")
            shutil.copy2(filepath, destination)

        except Exception as e:
            print(f"Error copying {filepath}: {e}")

print("Finished copying all non-scanned PDFs.")